# 4. Scam Call Detector Model Training
This notebook demonstrates how to train a Random Forest classifier using call metadata to identify fraudulent or spam voice calls.

### Objective:
- Build a structured classification pipeline for call metadata.
- Train the model using features like call frequency, duration, country code prefix, and prior spam reporting index.
- Export the trained pipeline to `call_classifier.pkl` in the python server directory.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Imports complete.")


## 2. Dataset Generation
We construct a synthetic dataset consisting of tabular call metadata:
- `call_duration` (seconds): Safe calls vary; scam calls are often brief or unusually long depending on scripts.
- `hourly_frequency`: Number of times the caller dialled other numbers in the last hour. (Scammers dial rapidly).
- `time_of_day` (hour): Night calls are higher risk.
- `spam_flags`: Total number of active user blocks on this caller ID in community directories.
- `carrier_reputation`: Rating from 1 (poor) to 5 (excellent).
- `international_prefix` (binary): Inbound calls from high-risk offshore VoIP ranges.

In [ ]:
np.random.seed(42)
num_samples = 250

duration_0 = np.random.normal(120, 45, num_samples)
freq_0 = np.random.poisson(2, num_samples)
hour_0 = np.random.randint(8, 20, num_samples)
flags_0 = np.random.poisson(0.1, num_samples)
rep_0 = np.random.choice([4, 5], num_samples, p=[0.3, 0.7])
intl_0 = np.random.choice([0, 1], num_samples, p=[0.95, 0.05])

duration_1 = np.random.normal(45, 20, num_samples)
freq_1 = np.random.poisson(85, num_samples)
hour_1 = np.random.randint(0, 24, num_samples)
flags_1 = np.random.poisson(12, num_samples)
rep_1 = np.random.choice([1, 2, 3], num_samples, p=[0.6, 0.3, 0.1])
intl_1 = np.random.choice([0, 1], num_samples, p=[0.4, 0.6])

df_0 = pd.DataFrame({
    'call_duration': duration_0,
    'hourly_frequency': freq_0,
    'time_of_day': hour_0,
    'spam_flags': flags_0,
    'carrier_reputation': rep_0,
    'international_prefix': intl_0,
    'label': 0
})

df_1 = pd.DataFrame({
    'call_duration': duration_1,
    'hourly_frequency': freq_1,
    'time_of_day': hour_1,
    'spam_flags': flags_1,
    'carrier_reputation': rep_1,
    'international_prefix': intl_1,
    'label': 1
})

df = pd.concat([df_0, df_1], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df.drop(columns=['label'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Data split: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows.")


## 3. Model Pipeline & Training
We initialize a **Random Forest Classifier** with 50 estimators (trees) to classify incoming calls based on metadata features. We fit the training data and measure model performance.

In [ ]:
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Validation Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


## 4. Exporting the Model
We export the trained Random Forest model to `call_classifier.pkl` in the `python/` directory.

In [ ]:
output_dir = '../python'
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'call_classifier.pkl')
joblib.dump(model, model_path)
print(f"Call classifier pipeline successfully exported to {model_path}")
